# Spark Smoke Test

Notebook de prueba para verificar que PySpark arranca, conecta al master si existe y ejecuta una operación simple.

In [1]:
import os
import socket
from urllib.parse import urlparse

from pyspark.sql import SparkSession

# Limpiar cualquier sesión anterior detenida
active_session = SparkSession.getActiveSession()
if active_session is not None:
    try:
        active_session.stop()
    except:
        pass

master_url = os.getenv("SPARK_MASTER", "spark://spark-master:7077")
use_remote_master = os.getenv("SPARK_USE_REMOTE_MASTER", "").strip().lower() in {"1", "true", "yes"}


def master_is_reachable(master: str) -> bool:
    parsed = urlparse(master)
    if parsed.scheme != "spark" or not parsed.hostname or not parsed.port:
        return False
    try:
        with socket.create_connection((parsed.hostname, parsed.port), timeout=2):
            return True
    except OSError:
        return False


spark_master = master_url if use_remote_master and master_is_reachable(master_url) else "local[*]"
if spark_master == "local[*]":
    print("Using local[*] so the smoke test does not depend on remote worker Python versions.")
else:
    print(f"Using remote Spark master: {spark_master}")

spark = (SparkSession.builder
         .appName("IABurnout-Spark-Smoke-Test")
         .master(spark_master)
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.sparkContext.master}")

Using local[*] so the smoke test does not depend on remote worker Python versions.
Spark version: 4.1.1
Spark master: local[*]


In [2]:
data = [
    (1, "alice", 10),
    (2, "bob", 20),
    (3, "carol", 30)
]

df = spark.createDataFrame(data, ["id", "name", "score"])
df.show()

summary = df.agg({"score": "avg", "id": "count"})
summary.show()

row_count = df.count()
avg_score = df.agg({"score": "avg"}).first()[0]

assert row_count == 3, f"Expected 3 rows, got {row_count}"
assert round(avg_score, 2) == 20.0, f"Expected average score 20.0, got {avg_score}"

print("Smoke test passed: Spark can create and process DataFrames.")

+---+-----+-----+
| id| name|score|
+---+-----+-----+
|  1|alice|   10|
|  2|  bob|   20|
|  3|carol|   30|
+---+-----+-----+

+----------+---------+
|avg(score)|count(id)|
+----------+---------+
|      20.0|        3|
+----------+---------+

Smoke test passed: Spark can create and process DataFrames.


In [3]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
